## Step1: dataset loading

In [ ]:
!PYTHONPATH=../../src python3 ../../src/datasets/main.py \
    dataset=imagenet \
    forget=filename \
    dataset.init_path="imagenet_example_data" \
    dataset.save_path="imagenet_example_split" \
    dataset.val_ratio=0.1 \
    forget.forget_filenames="[n01532829_1.JPEG]" \
    experiment_name="ggl"


## Step2: model loading

In [ ]:
!PYTHONPATH=../../src python3 -m train.main \
    dataset=imagenet \
    model=torchvision \
    model.model_name=resnet18 \
    +wandb=default \
    train_cfg=pretrained \
    trainer=trainer \
    dataset.load_dir="imagenet_example_split" \
    model.pretrained=True \
    model.num_classes=1000 \
    model.save_dir="artifacts/models"


In [ ]:
!PYTHONPATH=../../src python3 -m train.main \
    dataset=imagenet \
    model=torchvision \
    model.model_name=resnet18 \
    trainer=trainer \
    dataset.save_path="imagenet_example_split" \
    model.pretrained=true \
    trainer.model_save_dir="artifacts/models" \
    trainer.cfg.epochs=0 \
    +wandb=default \
    experiment_name=test

## Step 3: unlearning 

### neggrad 

In [ ]:
!PYTHONPATH=../../src python3 -m unlearning.main \
    dataset=imagenet \
    model=resnet18 \
    model=torchvision \
    unlearner=neggrad \
    dataset.save_path="imagenet_example_split" \
    model.model_name=resnet18 \
    model.original_model_ckpt_path="artifacts/models/resnet18_42_original.pt" \
    unlearner.cfg.lr=0.001 \
    unlearner.cfg.lr_decay_factor=0 \
    unlearner.cfg.epochs=1 \
    unlearner.evaluate=false \
    experiment_name=neggrad_unlearning \
    +wandb=default


### scrub

In [ ]:
!PYTHONPATH=../../src python3 -m unlearning.main \
    dataset=imagenet \
    model=resnet18 \
    model=torchvision \
    unlearner=scrub \
    dataset.save_path="imagenet_example_split" \
    model.model_name=resnet18 \
    model.original_model_ckpt_path="artifacts/models/resnet18_42_original.pt" \
    unlearner.cfg.lr=0.001 \
    unlearner.cfg.lr_decay_factor=0 \
    unlearner.cfg.min_epochs=0 \
    unlearner.cfg.momentum=0 \
    unlearner.cfg.weight_decay=0 \
    unlearner.cfg.max_epochs=1 \
    unlearner.cfg.alpha=1 \
    unlearner.cfg.gamma=0.5 \
    experiment_name=scrub_unlearning \
    +wandb=default 


## Step 4: GGL Reconstruction

### neggrad

In [ ]:
!PYTHONPATH=../../src python3 -m attacks.main_reconstructor \
    dataset=imagenet \
    attack=ggl \
    model=torchvision \
    dataset=imagenet \
    model.original_model_ckpt_path="artifacts/unlearn/neggrad_unlearning/unlearn/neggrad/resnet18_42_original.pt" \
    model.unlearned_model_ckpt_path="artifacts/unlearn/neggrad_unlearning/unlearn/neggrad/resnet18_42_unlearned.pt" \
    model.model_name=resnet18 \
    attack.unlearned_labels=[12] \
    dataset.save_path="imagenet_example_split" \
    attack.cfg.budget=500 \
    attack.lr=0.001 \
    +wandb=default \
    +wandb.extra_config.unlearning_method=neggrad \
    attack.cfg.initial_lr=1 \
    experiment_name=neggrad_reconstruction \
    +wandb.extra_config.epochs=1


### scrub

In [ ]:
!PYTHONPATH=../../src python3 -m attacks.main_reconstructor \
    dataset=imagenet \
    attack=ggl \
    model=torchvision \
    dataset=imagenet \
    model.original_model_ckpt_path="artifacts/unlearn/scrub_unlearning/unlearn/scrub/resnet18_42_original.pt" \
    model.unlearned_model_ckpt_path="artifacts/unlearn/scrub_unlearning/unlearn/scrub/resnet18_42_unlearned.pt" \
    model.model_name=resnet18 \
    dataset.save_path="imagenet_example_split" \
    attack.unlearned_labels=[12] \
    attack.cfg.budget=500 \
    attack.lr=0.001 \
    attack.cfg.unlearning_method=scrub \
    attack.unlearner.alpha=1 \
    attack.unlearner.gamma=0.5 \
    attack.unlearner.min_epochs=0 \
    attack.unlearner.max_epochs=1 \
    attack.cfg.initial_lr=1 \
    experiment_name=scrub_reconstruction \
    +wandb=default \
    +wandb.extra_config.unlearning_method=scrub \
    +wandb.extra_config.epochs=1
